In [1]:
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df = pd.read_csv('/content/drive/MyDrive/Classroom/DATN/job_descriptions.csv')[["description", "main_programming_languages", "key_technologies"]]
df

,description,main_programming_languages,key_technologies
0,Top 3 reasons to join us\n- Lương và chế độ hấ...,['Swift'],"['xCode', 'Realm', 'REST', 'JSON', 'Objective-C']"
1,Top 3 reasons to join us\n- 18+ days of annual...,"['Java', 'JavaScript', 'HTML/CSS']","['Spring Boot', 'Angular', 'React', 'Agile']"
2,LocalLife.Asia là nền tảng kết nối du khách vớ...,['TypeScript'],"['Next.js', 'Tailwind CSS']"
3,TymeX is an innovative digital banking develop...,['Python'],"['JIRA Service Management', 'Confluence', 'Ops..."
4,We are building an environment where personal ...,[],"['Agile', 'Automated Testing', 'Performance Te..."
...,...,...,...
3096,Mô tả công việc\n- Being part of the VN Softwa...,"['JavaScript', 'NodeJS']","['React', 'AWS', 'Google Cloud', 'Kafka', 'Doc..."
3097,Mô tả công việc\n- Develop and maintain high-p...,"['Python', 'Node.js']","['AWS', 'Serverless Architecture', 'CI/CD', 'I..."
3098,"• Thiết kế, lập trình và phát triển các websit...","['PHP', 'HTML', 'CSS', 'JavaScript', 'SQL']","['WordPress', 'SEO', 'Git']"
3099,Mô tả công việc\n1. Phát triển và tích hợp hệ ...,"['Python', 'C', 'JavaScript']","['Django', 'FastAPI', 'Flask', 'React', 'Flutt..."


In [4]:
label_name = ["B-LANG","I-LANG","B-TECH","I-TECH"]

In [5]:
import re
import ast
import pandas as pd

def convert_to_bio_format(text, prog_langs, techs):
    # Tách văn bản thành tokens
    tokens = re.findall(r'\w+|[^\w\s]', str(text))
    # Khởi tạo mảng labels toàn là 0 (đại diện cho nhãn 'O')
    labels = [0] * len(tokens)

    # Hàm gán nhãn cho danh sách từ khóa bằng ID
    def tag_keywords(keywords, b_tag_id, i_tag_id):
        for keyword in keywords:
            # Tách keyword thành các từ nhỏ
            kw_tokens = re.findall(r'\w+|[^\w\s]', str(keyword))
            kw_len = len(kw_tokens)

            # Khớp token (không phân biệt hoa/thường để tăng tỉ lệ chính xác)
            kw_tokens_lower = [k.lower() for k in kw_tokens]

            for i in range(len(tokens) - kw_len + 1):
                tokens_window_lower = [t.lower() for t in tokens[i:i+kw_len]]

                # Nếu khớp dãy token
                if tokens_window_lower == kw_tokens_lower:
                    labels[i] = b_tag_id           # Từ đầu tiên là B-
                    for j in range(1, kw_len):
                        labels[i+j] = i_tag_id     # Các từ sau là I-

    # Gỡ chuỗi "['Swift']" thành list thực thay vì chuỗi
    prog_langs_list = ast.literal_eval(prog_langs) if pd.notna(prog_langs) else []
    techs_list = ast.literal_eval(techs) if pd.notna(techs) else []

    # Gán ID nhãn cho từng loại
    # 1: 'B-LANG', 2: 'I-LANG'
    tag_keywords(prog_langs_list, 1, 2)
    # 3: 'B-TECH', 4: 'I-TECH'
    tag_keywords(techs_list, 3, 4)

    return {"tokens": tokens, "ner_tags": labels}


In [6]:
print(df.iloc[0]["description"], df.iloc[0]["main_programming_languages"], df.iloc[0]["key_technologies"])


print(convert_to_bio_format(df.iloc[0]["description"], df.iloc[0]["main_programming_languages"], df.iloc[0]["key_technologies"]))

Top 3 reasons to join us
- Lương và chế độ hấp dẫn, linh hoạt, cạnh tranh
- Cơ hội thăng tiến, phát triển trong công việc
- Môi trường thân thiện, trẻ trung, sáng tạo

Job description
- Xây dựng các ứng dụng Mobile Banking liên quan đến các mảng tài chính, Ngân hàng, Viễn thông trên nền tảng IOS
- Xây dựng các ứng dụng liên quan đến các phần tiện ích Ecommerce
- Nghiên cứu, tìm kiếm giải pháp về việc áp dụng các tính năng của thiết bị di động vào mảng tài chính, Ngân hàng, Viễn thông.
- Duy trì, hỗ trợ, nâng cấp các ứng dụng dịch vụ đã phát triển của Công ty
- Phát triển mới, nâng cấp ứng dụng cho các ngân hàng trong giai đoạn sắp tới.

Your skills and experience
- Tốt nghiệp Đại học/ Cao đẳng các chuyên ngành Công nghệ thông tin, Công nghệ phần mềm,..... hoặc Trường đào tạo lập trình viên Quốc Tế (NIIT, Aptech) trở lên
- Có từ 6 tháng kinh nghiệm trở lên
- Hiểu biết về xCode, Swift, có kinh nghiệm lập trình ứng dụng trên IOS
- Có hiểu biết về R

In [7]:
df["parse_bio"] = df.apply(lambda row: convert_to_bio_format(row["description"], row["main_programming_languages"], row["key_technologies"]), axis=1)

In [8]:
df

,description,main_programming_languages,key_technologies,parse_bio
0,Top 3 reasons to join us\n- Lương và chế độ hấ...,['Swift'],"['xCode', 'Realm', 'REST', 'JSON', 'Objective-C']","{'tokens': ['Top', '3', 'reasons', 'to', 'join..."
1,Top 3 reasons to join us\n- 18+ days of annual...,"['Java', 'JavaScript', 'HTML/CSS']","['Spring Boot', 'Angular', 'React', 'Agile']","{'tokens': ['Top', '3', 'reasons', 'to', 'join..."
2,LocalLife.Asia là nền tảng kết nối du khách vớ...,['TypeScript'],"['Next.js', 'Tailwind CSS']","{'tokens': ['LocalLife', '.', 'Asia', 'là', 'n..."
3,TymeX is an innovative digital banking develop...,['Python'],"['JIRA Service Management', 'Confluence', 'Ops...","{'tokens': ['TymeX', 'is', 'an', 'innovative',..."
4,We are building an environment where personal ...,[],"['Agile', 'Automated Testing', 'Performance Te...","{'tokens': ['We', 'are', 'building', 'an', 'en..."
...,...,...,...,...
3096,Mô tả công việc\n- Being part of the VN Softwa...,"['JavaScript', 'NodeJS']","['React', 'AWS', 'Google Cloud', 'Kafka', 'Doc...","{'tokens': ['Mô', 'tả', 'công', 'việc', '-', '..."
3097,Mô tả công việc\n- Develop and maintain high-p...,"['Python', 'Node.js']","['AWS', 'Serverless Architecture', 'CI/CD', 'I...","{'tokens': ['Mô', 'tả', 'công', 'việc', '-', '..."
3098,"• Thiết kế, lập trình và phát triển các websit...","['PHP', 'HTML', 'CSS', 'JavaScript', 'SQL']","['WordPress', 'SEO', 'Git']","{'tokens': ['•', 'Thiết', 'kế', ',', 'lập', 't..."
3099,Mô tả công việc\n1. Phát triển và tích hợp hệ ...,"['Python', 'C', 'JavaScript']","['Django', 'FastAPI', 'Flask', 'React', 'Flutt...","{'tokens': ['Mô', 'tả', 'công', 'việc', '1', '..."


In [9]:
checkpoint = "xlm-roberta-base"

label_names = ["O","B-LANG","I-LANG","B-TECH","I-TECH"]

id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

In [10]:
print(id2label)
print(label2id)

{0: 'O', 1: 'B-LANG', 2: 'I-LANG', 3: 'B-TECH', 4: 'I-TECH'}
{'O': 0, 'B-LANG': 1, 'I-LANG': 2, 'B-TECH': 3, 'I-TECH': 4}


In [11]:
from transformers import AutoTokenizer

# Create a tokenizer instance by loading the pre-trained checkpoint.
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [12]:
print(df.iloc[0]["parse_bio"]["tokens"])

['Top', '3', 'reasons', 'to', 'join', 'us', '-', 'Lương', 'và', 'chế', 'độ', 'hấp', 'dẫn', ',', 'linh', 'hoạt', ',', 'cạnh', 'tranh', '-', 'Cơ', 'hội', 'thăng', 'tiến', ',', 'phát', 'triển', 'trong', 'công', 'việc', '-', 'Môi', 'trường', 'thân', 'thiện', ',', 'trẻ', 'trung', ',', 'sáng', 'tạo', 'Job', 'description', '-', 'Xây', 'dư', '̣', 'ng', 'ca', '́', 'c', 'ư', '́', 'ng', 'du', '̣', 'ng', 'Mobile', 'Banking', 'liên', 'quan', 'đê', '́', 'n', 'ca', '́', 'c', 'ma', '̉', 'ng', 'ta', '̀', 'i', 'chi', '́', 'nh', ',', 'Ngân', 'hàng', ',', 'Viễn', 'thông', 'trên', 'nền', 'tảng', 'IOS', '-', 'Xây', 'dựng', 'các', 'ứng', 'dụng', 'liên', 'quan', 'đến', 'các', 'phần', 'tiện', 'ích', 'Ecommerce', '-', 'Nghiên', 'cư', '́', 'u', ',', 'ti', '̀', 'm', 'kiê', '́', 'm', 'gia', '̉', 'i', 'pha', '́', 'p', 'vê', '̀', 'viê', '̣', 'c', 'a', '́', 'p', 'du', '̣', 'ng', 'ca', '́', 'c', 'ti', '́', 'nh', 'năng', 'cu', '̉', 'a', 'thiê', '́', 't', 'bi', '̣', 'di', 'đô', '̣', 'ng', 'va', '̀', 'o', 'ma', '̉', 'ng'

In [13]:
# Tokenize the first training example from the dataset
token = tokenizer(df.iloc[0]["parse_bio"]["tokens"], is_split_into_words = True)

# Print the tokenizer object, the tokenized tokens, and the word IDs
print(token, '\n--------------------------------------------------------------------------------------\n',
      token.tokens(),'\n--------------------------------------------------------------------------------------\n',
      token.word_ids())

Token indices sequence length is longer than the specified maximum sequence length for this model (920 > 512). Running this sequence through the model will result in indexing errors


{'input_ids': [0, 4792, 138, 89397, 47, 33284, 1821, 20, 140976, 544, 12240, 6941, 53210, 10284, 6, 4, 42220, 9975, 6, 4, 26986, 21840, 20, 50096, 5869, 188498, 19743, 6, 4, 5152, 9442, 1000, 1871, 2735, 20, 67935, 14, 4373, 10807, 39469, 6, 4, 11824, 13375, 6, 4, 12107, 7217, 32664, 76811, 20, 112800, 61580, 6, 7035, 234, 377, 3309, 501, 6, 11479, 3309, 234, 115, 6, 7035, 234, 19745, 4932, 214, 8151, 2261, 99697, 3309, 653, 377, 3309, 501, 291, 6, 20586, 234, 308, 6, 10565, 17, 1658, 3309, 12203, 6, 4, 82165, 2508, 6, 4, 243139, 4225, 2479, 44565, 173947, 6, 62390, 20, 112800, 13291, 925, 13932, 2786, 8151, 2261, 1885, 925, 8192, 30276, 31037, 241, 67779, 20, 171917, 23823, 3309, 75, 6, 4, 1053, 6, 10565, 347, 200, 691, 3309, 347, 3529, 6, 20586, 17, 40681, 3309, 915, 6985, 6, 10565, 279, 691, 6, 7035, 501, 10, 3309, 915, 115, 6, 7035, 234, 377, 3309, 501, 1053, 3309, 12203, 5587, 314, 6, 20586, 10, 6117, 691, 3309, 808, 333, 6, 7035, 45, 29349, 6, 7035, 234, 307, 6, 10565, 36, 291, 6

In [14]:
def align_target(labels, word_ids):
    begin2inside = {
        1: 2,  # B-LANG (1) -> I-LANG (2)
        3: 4   # B-TECH (3) -> I-TECH (4)
    }

    align_labels = []
    last_word = None

    for word in word_ids:
        if word is None:
            label = -100
        elif word != last_word:
            label = labels[word]
        else:
            label = labels[word]
            if label in begin2inside:
                label = begin2inside[label]

        align_labels.append(label)
        last_word = word

    return align_labels

In [15]:
# Extract labels and word_ids
labels = df.iloc[0]["parse_bio"]['ner_tags']
word_ids = token.word_ids()

# Use the align_target function to align labels
aligned_target = align_target(labels, word_ids)

# Print tokenized tokens, original labels, and aligned labels
print(token.tokens(), '\n--------------------------------------------------------------------------------------\n',
      labels, '\n--------------------------------------------------------------------------------------\n',
      aligned_target)

['<s>', '▁Top', '▁3', '▁reasons', '▁to', '▁join', '▁us', '▁-', '▁Lương', '▁và', '▁chế', '▁độ', '▁hấp', '▁dẫn', '▁', ',', '▁linh', '▁hoạt', '▁', ',', '▁cạnh', '▁tranh', '▁-', '▁Cơ', '▁hội', '▁thăng', '▁tiến', '▁', ',', '▁phát', '▁triển', '▁trong', '▁công', '▁việc', '▁-', '▁Mô', 'i', '▁trường', '▁thân', '▁thiện', '▁', ',', '▁trẻ', '▁trung', '▁', ',', '▁sáng', '▁tạo', '▁Job', '▁description', '▁-', '▁Xây', '▁dư', '▁', '̣', '▁ng', '▁ca', '▁́', '▁c', '▁', 'ư', '▁́', '▁ng', '▁du', '▁', '̣', '▁ng', '▁Mobile', '▁Bank', 'ing', '▁liên', '▁quan', '▁đê', '▁́', '▁n', '▁ca', '▁́', '▁c', '▁ma', '▁', '̉', '▁ng', '▁ta', '▁', '̀', '▁i', '▁chi', '▁́', '▁nh', '▁', ',', '▁Ngân', '▁hàng', '▁', ',', '▁Viễn', '▁thông', '▁trên', '▁nền', '▁tảng', '▁', 'IOS', '▁-', '▁Xây', '▁dựng', '▁các', '▁ứng', '▁dụng', '▁liên', '▁quan', '▁đến', '▁các', '▁phần', '▁tiện', '▁ích', '▁E', 'commerce', '▁-', '▁Nghiên', '▁cư', '▁́', '▁u', '▁', ',', '▁ti', '▁', '̀', '▁m', '▁ki', 'ê', '▁́', '▁m', '▁gia', '▁', '̉', '▁i', '▁pha', '▁́', '

In [16]:
aligned_labels = [label_names[t] if t >= 0 else None for t in aligned_target]

# Loop through tokens and aligned labels and print them
for x, y in zip(token.tokens(), aligned_labels):
    print(f"{x}\t{y}")

<s>	None
▁Top	O
▁3	O
▁reasons	O
▁to	O
▁join	O
▁us	O
▁-	O
▁Lương	O
▁và	O
▁chế	O
▁độ	O
▁hấp	O
▁dẫn	O
▁	O
,	O
▁linh	O
▁hoạt	O
▁	O
,	O
▁cạnh	O
▁tranh	O
▁-	O
▁Cơ	O
▁hội	O
▁thăng	O
▁tiến	O
▁	O
,	O
▁phát	O
▁triển	O
▁trong	O
▁công	O
▁việc	O
▁-	O
▁Mô	O
i	O
▁trường	O
▁thân	O
▁thiện	O
▁	O
,	O
▁trẻ	O
▁trung	O
▁	O
,	O
▁sáng	O
▁tạo	O
▁Job	O
▁description	O
▁-	O
▁Xây	O
▁dư	O
▁	O
̣	O
▁ng	O
▁ca	O
▁́	O
▁c	O
▁	O
ư	O
▁́	O
▁ng	O
▁du	O
▁	O
̣	O
▁ng	O
▁Mobile	O
▁Bank	O
ing	O
▁liên	O
▁quan	O
▁đê	O
▁́	O
▁n	O
▁ca	O
▁́	O
▁c	O
▁ma	O
▁	O
̉	O
▁ng	O
▁ta	O
▁	O
̀	O
▁i	O
▁chi	O
▁́	O
▁nh	O
▁	O
,	O
▁Ngân	O
▁hàng	O
▁	O
,	O
▁Viễn	O
▁thông	O
▁trên	O
▁nền	O
▁tảng	O
▁	O
IOS	O
▁-	O
▁Xây	O
▁dựng	O
▁các	O
▁ứng	O
▁dụng	O
▁liên	O
▁quan	O
▁đến	O
▁các	O
▁phần	O
▁tiện	O
▁ích	O
▁E	O
commerce	O
▁-	O
▁Nghiên	O
▁cư	O
▁́	O
▁u	O
▁	O
,	O
▁ti	O
▁	O
̀	O
▁m	O
▁ki	O
ê	O
▁́	O
▁m	O
▁gia	O
▁	O
̉	O
▁i	O
▁pha	O
▁́	O
▁p	O
▁vê	O
▁	O
̀	O
▁vi	O
ê	O
▁	O
̣	O
▁c	O
▁a	O
▁́	O
▁p	O
▁du	O
▁	O
̣	O
▁ng	O
▁ca	O
▁́	O
▁c	O
▁ti	O
▁́	O
▁nh	O
▁năng	O
▁cu	O
▁	O
̉	O
▁a	O


In [17]:
print(word_ids)

[None, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 13, 14, 15, 16, 16, 17, 18, 19, 20, 21, 22, 23, 24, 24, 25, 26, 27, 28, 29, 30, 31, 31, 32, 33, 34, 35, 35, 36, 37, 38, 38, 39, 40, 41, 42, 43, 44, 45, 46, 46, 47, 48, 49, 50, 51, 51, 52, 53, 54, 55, 55, 56, 57, 58, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 68, 69, 70, 71, 71, 72, 73, 74, 75, 76, 76, 77, 78, 79, 79, 80, 81, 82, 83, 84, 85, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 99, 100, 101, 102, 103, 104, 105, 105, 106, 107, 107, 108, 109, 109, 110, 111, 112, 113, 113, 114, 115, 116, 117, 118, 119, 119, 120, 120, 121, 121, 122, 123, 124, 125, 126, 127, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 137, 138, 139, 139, 140, 141, 142, 143, 143, 144, 145, 146, 146, 147, 148, 149, 149, 150, 151, 152, 152, 153, 154, 155, 155, 156, 157, 158, 159, 160, 160, 161, 162, 163, 163, 164, 165, 166, 166, 167, 168, 169, 170, 170, 171, 171, 172, 173, 173, 174, 175, 175, 176, 176, 177, 178, 179, 180, 181, 182, 183, 184,

In [18]:
print(labels)

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [19]:
# Define fake input data
words = ['<s>', '▁Top', '▁3', '▁reasons', '▁to', '▁join', '▁us', '▁-', '▁Lương', '▁và', '▁chế', '▁độ', '▁hấp', '▁dẫn', '▁', ',', '▁linh', '▁hoạt', '▁', ',', '▁cạnh', '▁tranh', '▁-', '▁Cơ', '▁hội', '▁thăng', '▁tiến', '▁', ',', '▁phát', '▁triển', '▁trong', '▁công', '▁việc', '▁-', '▁Mô', 'i', '▁trường', '▁thân', '▁thiện', '▁', ',', '▁trẻ', '▁trung', '▁', ',', '▁sáng', '▁tạo', '▁Job', '▁description', '▁-', '▁Xây', '▁dư', '▁', '̣', '▁ng', '▁ca', '▁́', '▁c', '▁', 'ư', '▁́', '▁ng', '▁du', '▁', '̣', '▁ng', '▁Mobile', '▁Bank', 'ing', '▁liên', '▁quan', '▁đê', '▁́', '▁n', '▁ca', '▁́', '▁c', '▁ma', '▁', '̉', '▁ng', '▁ta', '▁', '̀', '▁i', '▁chi', '▁́', '▁nh', '▁', ',', '▁Ngân', '▁hàng', '▁', ',', '▁Viễn', '▁thông', '▁trên', '▁nền', '▁tảng', '▁', 'IOS', '▁-', '▁Xây', '▁dựng', '▁các', '▁ứng', '▁dụng', '▁liên', '▁quan', '▁đến', '▁các', '▁phần', '▁tiện', '▁ích', '▁E', 'commerce', '▁-', '▁Nghiên', '▁cư', '▁́', '▁u', '▁', ',', '▁ti', '▁', '̀', '▁m', '▁ki', 'ê', '▁́', '▁m', '▁gia', '▁', '̉', '▁i', '▁pha', '▁́', '▁p', '▁vê', '▁', '̀', '▁vi', 'ê', '▁', '̣', '▁c', '▁a', '▁́', '▁p', '▁du', '▁', '̣', '▁ng', '▁ca', '▁́', '▁c', '▁ti', '▁́', '▁nh', '▁năng', '▁cu', '▁', '̉', '▁a', '▁thi', 'ê', '▁́', '▁t', '▁bi', '▁', '̣', '▁di', '▁đô', '▁', '̣', '▁ng', '▁va', '▁', '̀', '▁o', '▁ma', '▁', '̉', '▁ng', '▁ta', '▁', '̀', '▁i', '▁chi', '▁́', '▁nh', '▁', ',', '▁Ngân', '▁hàng', '▁', ',', '▁Viễn', '▁thông', '▁', '.', '▁-', '▁Duy', '▁tri', '▁', '̀', '▁', ',', '▁hô', '▁', '̃', '▁trơ', '▁', '̣', '▁', ',', '▁nâng', '▁câ', '▁́', '▁p', '▁ca', '▁́', '▁c', '▁', 'ư', '▁́', '▁ng', '▁du', '▁', '̣', '▁ng', '▁di', '▁', '̣', '▁ch', '▁vu', '▁', '̣', '▁đa', '▁', '̃', '▁pha', '▁́', '▁t', '▁tri', 'ê', '▁', '̉', '▁n', '▁cu', '▁', '̉', '▁a', '▁Công', '▁ty', '▁-', '▁Phát', '▁triển', '▁mới', '▁', ',', '▁nâng', '▁cấp', '▁ứng', '▁dụng', '▁cho', '▁các', '▁ngân', '▁hàng', '▁trong', '▁giai', '▁đoạn', '▁sắp', '▁tới', '▁', '.', '▁Your', '▁skills', '▁and', '▁experience', '▁-', '▁T', 'ốt', '▁nghiệp', '▁Đại', '▁học', '▁/', '▁Cao', '▁đẳng', '▁các', '▁chuyên', '▁ngành', '▁Công', '▁nghệ', '▁thông', '▁tin', '▁', ',', '▁Công', '▁nghệ', '▁phần', '▁mềm', '▁', ',', '▁', '.', '▁', '.', '▁', '.', '▁', '.', '▁', '.', '▁hoặc', '▁Trường', '▁đào', '▁tạo', '▁lập', '▁trình', '▁viên', '▁Quốc', '▁Tế', '▁(', '▁NI', 'IT', '▁', ',', '▁Ap', 'tech', '▁)', '▁trở', '▁lên', '▁-', '▁Có', '▁từ', '▁6', '▁tháng', '▁kinh', '▁nghiệm', '▁trở', '▁lên', '▁-', '▁Hiểu', '▁biết', '▁về', '▁x', 'Code', '▁', ',', '▁Swift', '▁', ',', '▁có', '▁kinh', '▁nghiệm', '▁lập', '▁trình', '▁ứng', '▁dụng', '▁trên', '▁', 'IOS', '▁-', '▁Có', '▁hiểu', '▁biết', '▁về', '▁Real', 'm', '▁', ',', '▁giao', '▁tiếp', '▁client', '▁–', '▁server', '▁thông', '▁qua', '▁R', 'EST', '▁và', '▁J', 'SON', '▁-', '▁Có', '▁kiến', '▁thức', '▁về', '▁Object', 'ive', '▁–', '▁C', '▁là', '▁1', '▁lợi', '▁thế', '▁', ',', '▁chịu', '▁khó', '▁học', '▁hỏi', '▁', ',', '▁tìm', '▁hiểu', '▁kiến', '▁thức', '▁mới', '▁-', '▁Ưu', '▁tiên', '▁ứng', '▁viên', '▁có', '▁kinh', '▁nghiệm', '▁làm', '▁về', '▁lĩnh', '▁vực', '▁tài', '▁chính', '▁', ',', '▁ngân', '▁hàng', '▁Why', '▁you', "▁'", '▁', 'll', '▁love', '▁working', '▁here', '▁1', '▁', '.', '▁Chế', '▁độ', '▁lương', '▁&', '▁thưởng', '▁hấp', '▁dẫn', '▁:', '▁-', '▁M', 'ức', '▁lương', '▁thỏa', '▁thuận', '▁theo', '▁năng', '▁lực', '▁', ',', '▁đánh', '▁giá', '▁năng', '▁lực', '▁hàng', '▁năm', '▁-', '▁Chế', '▁độ', '▁thưởng', '▁phong', '▁phú', '▁và', '▁hấp', '▁dẫn', '▁(', '▁Theo', '▁quy', '▁định', '▁và', '▁chính', '▁sách', '▁công', '▁ty', '▁)', '▁-', '▁Hỗ', '▁trợ', '▁ăn', '▁sáng', '▁miễn', '▁phí', '▁-', '▁Hỗ', '▁trợ', '▁ăn', '▁trưa', '▁:', '▁50', '▁', '.', '▁000', 'đ', '▁/', '▁ngày', '▁-', '▁Hỗ', '▁trợ', '▁gửi', '▁xe', '▁', ',', '▁xăng', '▁xe', '▁', ',', '▁điện', '▁thoại', '▁(', '▁tùy', '▁vị', '▁trí', '▁)', '▁-', '▁Hỗ', '▁trợ', '▁trang', '▁điểm', '▁cho', '▁CB', 'NV', '▁nữ', '▁-', '▁Qu', 'à', '▁tặng', '▁các', '▁ngày', '▁lễ', '▁trong', '▁năm', '▁2', '▁', '.', '▁Bảo', '▁hiểm', '▁&', '▁chăm', '▁sóc', '▁sức', '▁khỏe', '▁toàn', '▁diện', '▁:', '▁-', '▁BH', 'XH', '▁', ',', '▁BH', 'YT', '▁', ',', '▁BH', 'TN', '▁theo', '▁pháp', '▁luật', '▁hiện', '▁hành', '▁-', '▁G', 'ói', '▁bảo', '▁hiểm', '▁sức', '▁khỏe', '▁cao', '▁cấp', '▁24', '▁/', '▁7', '▁mua', '▁cho', '▁nhân', '▁viên', '▁&', '▁người', '▁thân', '▁nhân', '▁viên', '▁-', '▁Khám', '▁sức', '▁khỏe', '▁định', '▁kỳ', '▁hàng', '▁năm', '▁3', '▁', '.', '▁Mô', 'i', '▁trường', '▁làm', '▁việc', '▁hiện', '▁đại', '▁:', '▁-', '▁Cung', '▁cấp', '▁máy', '▁tính', '▁&', '▁thiết', '▁bị', '▁làm', '▁việc', '▁hiện', '▁đại', '▁-', '▁Văn', '▁phòng', '▁làm', '▁việc', '▁hiện', '▁đại', '▁', ',', '▁thiết', '▁bị', '▁làm', '▁việc', '▁Hi', '▁-', '▁tech', '▁-', '▁Không', '▁gian', '▁ăn', '▁nhẹ', '▁miễn', '▁phí', '▁(', '▁nước', '▁uống', '▁', ',', '▁trà', '▁cafe', '▁', ',', '▁hoa', '▁quả', '▁', ',', '▁sữa', '▁chua', '▁)', '▁4', '▁', '.', '▁Phát', '▁triển', '▁nghề', '▁nghiệp', '▁:', '▁-', '▁Cơ', '▁hội', '▁tiếp', '▁cận', '▁với', '▁những', '▁công', '▁nghệ', '▁mới', '▁', ',', '▁những', '▁dự', '▁án', '▁quy', '▁mô', '▁lớn', '▁-', '▁Làm', '▁việc', '▁cùng', '▁đội', '▁ngũ', '▁hơn', '▁2000', '▁nhân', '▁sự', '▁tài', '▁năng', '▁', ',', '▁có', '▁chuyên', '▁môn', '▁giỏi', '▁', ',', '▁dày', '▁d', 'ặ', 'n', '▁kinh', '▁nghiệm', '▁', ',', '▁tư', '▁duy', '▁chia', '▁sẻ', '▁-', '▁Được', '▁tài', '▁trợ', '▁kinh', '▁phí', '▁tham', '▁gia', '▁các', '▁chương', '▁trình', '▁đào', '▁tạo', '▁nâng', '▁cao', '▁năng', '▁lực', '▁-', '▁Công', '▁ty', '▁thành', '▁lập', '▁gần', '▁20', '▁năm', '▁với', '▁các', '▁sản', '▁phẩm', '▁&', '▁dịch', '▁vụ', '▁đã', '▁được', '▁k', 'hẳng', '▁định', '▁vị', '▁thế', '▁trên', '▁thị', '▁trường', '▁5', '▁', '.', '▁Hoạt', '▁động', '▁ngoại', '▁khóa', '▁phong', '▁phú', '▁:', '▁-', '▁Văn', '▁hóa', '▁công', '▁ty', '▁đặc', '▁sắc', '▁với', '▁nhiều', '▁hoạt', '▁động', '▁đoàn', '▁thể', '▁được', '▁quan', '▁tâm', '▁đầu', '▁tư', '▁:', '▁Team', '▁building', '▁', ',', '▁Ngh', 'ỉ', '▁mát', '▁(', '▁trong', '▁nước', '▁và', '▁nước', '▁ngoài', '▁)', '▁', ',', '▁20', '▁/', '▁10', '▁', ',', '▁Year', '▁End', '▁Party', '▁', ',', '▁hoạt', '▁động', '▁thiện', '▁nguyện', '▁', ',', '▁', '…', '▁-', '▁Các', '▁câu', '▁lạc', '▁bộ', '▁:', '▁C', 'LB', '▁Cầu', '▁lông', '▁', ',', '▁C', 'LB', '▁Bó', 'ng', '▁đá', '▁', ',', '▁C', 'LB', '▁Yoga', '▁', ',', '▁C', 'LB', '▁Đ', 'iền', '▁kinh', '▁', ',', '▁C', 'LB', '▁B', 'ơ', 'i', '▁Chúng', '▁tôi', '▁sẽ', '▁liên', '▁hệ', '▁qua', '▁điện', '▁thoại', '▁với', '▁những', '▁CV', '▁phù', '▁hợp', '▁trong', '▁vòng', '▁7', '▁ngày', '▁làm', '▁việc', '▁kể', '▁từ', '▁ngày', '▁ứng', '▁tuyển', '▁', '.', '▁Lưu', '▁ý', '▁:', '▁Công', '▁ty', '▁đang', '▁tuyển', '▁tại', '▁2', '▁văn', '▁phòng', '▁Hà', '▁Nội', '▁và', '▁Tp', '▁', '.', '▁Hồ', '▁Chí', '▁Minh', '▁', ',', '▁ứng', '▁viên', '▁vui', '▁lòng', '▁ghi', '▁rõ', '▁địa', '▁chỉ', '▁trên', '▁CV', '▁', '.', '</s>']
word_ids = [None, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 13, 14, 15, 16, 16, 17, 18, 19, 20, 21, 22, 23, 24, 24, 25, 26, 27, 28, 29, 30, 31, 31, 32, 33, 34, 35, 35, 36, 37, 38, 38, 39, 40, 41, 42, 43, 44, 45, 46, 46, 47, 48, 49, 50, 51, 51, 52, 53, 54, 55, 55, 56, 57, 58, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 68, 69, 70, 71, 71, 72, 73, 74, 75, 76, 76, 77, 78, 79, 79, 80, 81, 82, 83, 84, 85, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 99, 100, 101, 102, 103, 104, 105, 105, 106, 107, 107, 108, 109, 109, 110, 111, 112, 113, 113, 114, 115, 116, 117, 118, 119, 119, 120, 120, 121, 121, 122, 123, 124, 125, 126, 127, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 137, 138, 139, 139, 140, 141, 142, 143, 143, 144, 145, 146, 146, 147, 148, 149, 149, 150, 151, 152, 152, 153, 154, 155, 155, 156, 157, 158, 159, 160, 160, 161, 162, 163, 163, 164, 165, 166, 166, 167, 168, 169, 170, 170, 171, 171, 172, 173, 173, 174, 175, 175, 176, 176, 177, 178, 179, 180, 181, 182, 183, 184, 184, 185, 186, 187, 188, 188, 189, 190, 191, 191, 192, 193, 194, 194, 195, 196, 196, 197, 198, 199, 200, 200, 201, 201, 202, 203, 204, 204, 205, 206, 207, 208, 209, 210, 211, 212, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 226, 227, 228, 229, 230, 231, 232, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 246, 247, 248, 249, 250, 251, 251, 252, 252, 253, 253, 254, 254, 255, 255, 256, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 267, 268, 268, 269, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 286, 287, 287, 288, 289, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 298, 299, 300, 301, 302, 303, 304, 304, 305, 305, 306, 307, 308, 309, 310, 311, 312, 313, 313, 314, 315, 315, 316, 317, 318, 319, 320, 321, 321, 322, 323, 324, 325, 326, 327, 328, 328, 329, 330, 331, 332, 333, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 353, 354, 355, 356, 357, 358, 359, 359, 360, 361, 362, 363, 364, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 374, 375, 376, 377, 378, 379, 380, 381, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 421, 422, 422, 423, 424, 425, 426, 427, 428, 429, 430, 430, 431, 432, 433, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 447, 448, 449, 450, 450, 451, 452, 453, 454, 455, 456, 457, 458, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 470, 471, 471, 472, 472, 473, 473, 474, 474, 475, 476, 477, 478, 479, 480, 481, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 509, 510, 510, 511, 512, 513, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 524, 525, 526, 527, 528, 529, 530, 531, 532, 533, 534, 535, 536, 536, 537, 538, 539, 540, 541, 542, 543, 544, 545, 546, 547, 548, 549, 550, 551, 552, 553, 554, 554, 555, 556, 557, 557, 558, 559, 560, 560, 561, 562, 563, 564, 565, 565, 566, 567, 568, 569, 570, 571, 572, 573, 574, 575, 576, 577, 578, 579, 580, 581, 581, 582, 583, 584, 585, 586, 587, 588, 589, 590, 591, 592, 593, 594, 595, 596, 597, 598, 599, 600, 600, 601, 602, 603, 604, 605, 605, 606, 607, 607, 607, 608, 609, 610, 610, 611, 612, 613, 614, 615, 616, 617, 618, 619, 620, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 631, 632, 633, 634, 635, 636, 637, 638, 639, 640, 641, 642, 643, 644, 645, 646, 647, 648, 649, 649, 650, 651, 652, 653, 654, 655, 656, 657, 657, 658, 659, 660, 661, 662, 663, 664, 665, 666, 667, 668, 669, 670, 671, 672, 673, 674, 675, 676, 677, 678, 679, 680, 681, 682, 683, 684, 685, 686, 686, 687, 687, 688, 689, 690, 691, 692, 693, 694, 695, 696, 696, 697, 698, 699, 700, 700, 701, 702, 703, 704, 704, 705, 706, 707, 708, 709, 709, 710, 710, 711, 712, 713, 714, 715, 716, 717, 717, 718, 719, 720, 720, 721, 721, 722, 722, 723, 724, 724, 725, 725, 726, 727, 727, 728, 728, 729, 729, 730, 731, 731, 732, 732, 733, 733, 733, 734, 735, 736, 737, 738, 739, 740, 741, 742, 743, 744, 745, 746, 747, 748, 749, 750, 751, 752, 753, 754, 755, 756, 757, 758, 758, 759, 760, 761, 762, 763, 764, 765, 766, 767, 768, 769, 770, 771, 772, 773, 774, 774, 775, 776, 777, 778, 778, 779, 780, 781, 782, 783, 784, 785, 786, 787, 788, 789, 789, None]
labels = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

# Use the align_target function to align labels
aligned_target = align_target(labels, word_ids)

# Create a list of aligned labels using label names
aligned_labels = [label_names[t] if t >= 0 else None for t in aligned_target]

# Loop through words and aligned labels and print them
for x, y in zip(words, aligned_labels):
    print(f"{x}\t{y}")

<s>	None
▁Top	O
▁3	O
▁reasons	O
▁to	O
▁join	O
▁us	O
▁-	O
▁Lương	O
▁và	O
▁chế	O
▁độ	O
▁hấp	O
▁dẫn	O
▁	O
,	O
▁linh	O
▁hoạt	O
▁	O
,	O
▁cạnh	O
▁tranh	O
▁-	O
▁Cơ	O
▁hội	O
▁thăng	O
▁tiến	O
▁	O
,	O
▁phát	O
▁triển	O
▁trong	O
▁công	O
▁việc	O
▁-	O
▁Mô	O
i	O
▁trường	O
▁thân	O
▁thiện	O
▁	O
,	O
▁trẻ	O
▁trung	O
▁	O
,	O
▁sáng	O
▁tạo	O
▁Job	O
▁description	O
▁-	O
▁Xây	O
▁dư	O
▁	O
̣	O
▁ng	O
▁ca	O
▁́	O
▁c	O
▁	O
ư	O
▁́	O
▁ng	O
▁du	O
▁	O
̣	O
▁ng	O
▁Mobile	O
▁Bank	O
ing	O
▁liên	O
▁quan	O
▁đê	O
▁́	O
▁n	O
▁ca	O
▁́	O
▁c	O
▁ma	O
▁	O
̉	O
▁ng	O
▁ta	O
▁	O
̀	O
▁i	O
▁chi	O
▁́	O
▁nh	O
▁	O
,	O
▁Ngân	O
▁hàng	O
▁	O
,	O
▁Viễn	O
▁thông	O
▁trên	O
▁nền	O
▁tảng	O
▁	O
IOS	O
▁-	O
▁Xây	O
▁dựng	O
▁các	O
▁ứng	O
▁dụng	O
▁liên	O
▁quan	O
▁đến	O
▁các	O
▁phần	O
▁tiện	O
▁ích	O
▁E	O
commerce	O
▁-	O
▁Nghiên	O
▁cư	O
▁́	O
▁u	O
▁	O
,	O
▁ti	O
▁	O
̀	O
▁m	O
▁ki	O
ê	O
▁́	O
▁m	O
▁gia	O
▁	O
̉	O
▁i	O
▁pha	O
▁́	O
▁p	O
▁vê	O
▁	O
̀	O
▁vi	O
ê	O
▁	O
̣	O
▁c	O
▁a	O
▁́	O
▁p	O
▁du	O
▁	O
̣	O
▁ng	O
▁ca	O
▁́	O
▁c	O
▁ti	O
▁́	O
▁nh	O
▁năng	O
▁cu	O
▁	O
̉	O
▁a	O


In [20]:
def tokenize_fn(batch):
    # Extract tokens and ner_tags from the 'parse_bio' column for the current batch
    tokens_batch = [item['tokens'] for item in batch['parse_bio']]
    ner_tags_batch = [item['ner_tags'] for item in batch['parse_bio']]

    # Tokenize the input batch using the extracted tokens
    tokenized_inputs = tokenizer(tokens_batch, truncation=True, max_length=512, is_split_into_words=True)

    # Initialize a list to store aligned targets for each example in the batch
    aligned_targets_batch = []

    # Iterate through each example's ner_tags and align them
    for i, labels in enumerate(ner_tags_batch):
        # Extract the word_ids for the current example from the tokenized inputs
        word_ids = tokenized_inputs.word_ids(i)

        # Use the align_target function to align the labels
        aligned_targets_batch.append(align_target(labels, word_ids))

    # Add the aligned labels to the tokenized inputs under the key "labels"
    tokenized_inputs["labels"] = aligned_targets_batch

    # Return the tokenized inputs, including aligned labels
    return tokenized_inputs

In [21]:
from datasets import Dataset

# Create a Hugging Face Dataset from the pandas DataFrame
dataset = Dataset.from_pandas(df)

# Apply the tokenize_fn to the dataset
tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=list(df.columns))

# Split the tokenized_dataset into training and testing sets
splits = tokenized_dataset.train_test_split(test_size=0.2)
train_dataset = splits['train']
eval_dataset = splits['test']

Map:   0%|          | 0/3101 [00:00<?, ? examples/s]

In [22]:
from transformers import DataCollatorForTokenClassification

# Create a DataCollatorForTokenClassification object
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Testing data using the data collator
batch = data_collator([tokenized_dataset[i] for i in range(2)])

# Display the resulting batch
batch

{'input_ids': tensor([[     0,   4792,    138,  ...,      6,      4,      2],
        [     0,   4792,    138,  ..., 173591,  39272,      2]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]]), 'labels': tensor([[-100,    0,    0,  ...,    0,    0, -100],
        [-100,    0,    0,  ...,    0,    0, -100]])}

In [23]:
!pip install evaluate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.6 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=bf0684f5b43c08ee8e8df7fdc2e777d76a6b9adb495b5f9ee23015174076eb87
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [24]:
from evaluate import load

# Load the seqeval metric which can evaluate NER and other sequence tasks
metric = load("seqeval")

# Example usage: compute metric on a sample predictions and reference list
# predictions and references should be a list of lists containing predicted and true token labels

# List of List Input
metric.compute(predictions = [['O' , 'B-LANG' , 'I-LANG']],
               references = [['O' , 'B-TECH' , 'I-TECH']])

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'LANG': {'precision': np.float64(0.0),
  'recall': np.float64(0.0),
  'f1': np.float64(0.0),
  'number': np.int64(0)},
 'TECH': {'precision': np.float64(0.0),
  'recall': np.float64(0.0),
  'f1': np.float64(0.0),
  'number': np.int64(1)},
 'overall_precision': np.float64(0.0),
 'overall_recall': np.float64(0.0),
 'overall_f1': np.float64(0.0),
 'overall_accuracy': 0.3333333333333333}

In [25]:
# Function to compute evaluation metrics from model logits and true labels
def compute_metrics(logits_and_labels):

  # Unpack the logits and labels
  logits, labels = logits_and_labels

  # Get predictions from the logits
  predictions = np.argmax(logits, axis=-1)

  # Remove ignored index (special tokens)
  str_labels = [
    [label_names[t] for t in label if t!=-100] for label in labels
  ]

  str_preds = [
    [label_names[p] for (p, t) in zip(prediction, label) if t != -100]
    for prediction, label in zip(predictions, labels)
  ]

  # Compute metrics
  results = metric.compute(predictions=str_preds, references=str_labels)

  # Extract key metrics
  return {
    "precision": results["overall_precision"],
    "recall": results["overall_recall"],
    "f1": results["overall_f1"],
    "accuracy": results["overall_accuracy"]
  }

In [26]:
# Create mapping from label ID to label string name
id2label = {k: v for k, v in enumerate(label_names)}

# Create reverse mapping from label name to label ID
label2id = {v: k for k, v in enumerate(label_names)}

print(id2label , '\n--------------------\n' , label2id)

{0: 'O', 1: 'B-LANG', 2: 'I-LANG', 3: 'B-TECH', 4: 'I-TECH'} 
--------------------
 {'O': 0, 'B-LANG': 1, 'I-LANG': 2, 'B-TECH': 3, 'I-TECH': 4}


In [27]:
# Load pretrained token classification model from Transformers
from transformers import AutoModelForTokenClassification

# Initialize model object with pretrained weights
model = AutoModelForTokenClassification.from_pretrained(
  checkpoint,

  # Pass in label mappings
  id2label=id2label,
  label2id=label2id
)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [28]:
# Configure training arguments using TrainigArguments class
from transformers import TrainingArguments

training_args = TrainingArguments(
  # Location to save fine-tuned model
  output_dir = "fine_tuned_model",

  # Evaluate each epoch
  eval_strategy = "epoch",

  # Learning rate for Adam optimizer
  learning_rate = 2e-5,

  # Batch sizes for training and evaluation
  per_device_train_batch_size = 16,
  per_device_eval_batch_size = 16,

  # Number of training epochs
  num_train_epochs = 3,

  # L2 weight decay regularization
  weight_decay = 0.01
)

In [29]:
# Initialize Trainer object for model training
from transformers import Trainer

trainer = Trainer(
  # Model to train
  model=model,

  # Training arguments
  args=training_args,

  # Training and validation datasets
  train_dataset=train_dataset,
  eval_dataset=eval_dataset,

  # Custom metric function
  compute_metrics=compute_metrics,

  # Data collator
  data_collator=data_collator
)

In [30]:
import numpy as np

In [31]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.046015,0.549370,0.696869,0.614391,0.981420
2,No log,0.037141,0.688062,0.730846,0.708809,0.985374
3,No log,0.036554,0.683134,0.786143,0.731027,0.985632


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=465, training_loss=0.05997222162062122, metrics={'train_runtime': 907.6327, 'train_samples_per_second': 8.197, 'train_steps_per_second': 0.512, 'total_flos': 1944100598169600.0, 'train_loss': 0.05997222162062122, 'epoch': 3.0})

In [32]:
trainer.save_model('fine_tuned_model_NER')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [34]:
import shutil

# Đường dẫn nguồn và đường dẫn đích trên Drive
source_dir = '/content/fine_tuned_model'
destination_dir = '/content/drive/MyDrive/Classroom/DATN/Model'

# Sao chép thư mục, cho phép thư mục đích tồn tại
shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)

'/content/drive/MyDrive/Classroom/DATN/Model'

### Load the fine-tuned model and tokenizer for inference

In [37]:
!pip install sentencepiece

In [41]:
from transformers import pipeline

ner = pipeline(
    'token-classification',
    model = 'fine_tuned_model_NER',
    aggregation_strategy = 'simple' ,
    device = 0
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [44]:
ner('Cử nhân chuyên ngành Công nghệ thông tin, Điện tử viễn thông, Tài chính, Ngân hàng, Kinh tế hoặc tương đương Có từ 03 năm kinh nghiệm phát triển hệ thống backend bằng Python. Hiểu biết vững về kiến trúc Microservices. Thành thạo Docker và có kinh nghiệm triển khai ứng dụng trên k8s (k8s). Có kinh nghiệm làm việc với RESTful API, Database (SQL/NoSQL). Ưu tiên: Có hiểu biết hoặc kinh nghiệm về Frontend (ReactJS) là một lợi thế. Kỹ năng phân tích, giải quyết vấn đề và làm việc nhóm tốt. Tư duy chủ động, tinh thần học hỏi và trách nhiệm cao.')

[{'entity_group': 'LANG',
  'score': np.float32(0.98084086),
  'word': 'Python',
  'start': 167,
  'end': 173},
 {'entity_group': 'TECH',
  'score': np.float32(0.8814525),
  'word': 'Microservices',
  'start': 203,
  'end': 216},
 {'entity_group': 'TECH',
  'score': np.float32(0.9636601),
  'word': 'Docker',
  'start': 229,
  'end': 235}]

### Explicitly load model and tokenizer, then create pipeline

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# Define the path where the model was saved
model_path = "fine_tuned_model_NER"

# Load the tokenizer
loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)

# Define label mappings (if not already defined or to ensure consistency)
label_names = ["O","B-LANG","I-LANG","B-TECH","I-TECH"]
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

# Load the model, ensuring id2label and label2id are set for proper label mapping
loaded_model = AutoModelForTokenClassification.from_pretrained(
    model_path,
    id2label=id2label,
    label2id=label2id
)

print("Model and tokenizer loaded successfully using AutoModel and AutoTokenizer!")

# Create a pipeline using the explicitly loaded model and tokenizer
explicit_ner_pipeline = pipeline(
    'token-classification',
    model=loaded_model,
    tokenizer=loaded_tokenizer,
    aggregation_strategy='max',
    device=0 # Use 0 for GPU if available, -1 for CPU
)

# Example usage with the explicit pipeline
sample_text = "Cử nhân chuyên ngành Công nghệ thông tin, Điện tử viễn thông, Tài chính, Ngân hàng, Kinh tế hoặc tương đương Có từ 03 năm kinh nghiệm phát triển hệ thống backend bằng Python. Hiểu biết vững về kiến trúc Microservices. Thành thạo Docker và có kinh nghiệm triển khai ứng dụng trên k8s (k8s). Có kinh nghiệm làm việc với RESTful API, Database (SQL/NoSQL). Ưu tiên: Có hiểu biết hoặc kinh nghiệm về Frontend (ReactJS) là một lợi thế. Kỹ năng phân tích, giải quyết vấn đề và làm việc nhóm tốt. Tư duy chủ động, tinh thần học hỏi và trách nhiệm cao."
predictions = explicit_ner_pipeline(sample_text)

# Print the predictions
print(predictions)

### Giải pháp khắc phục lỗi tách từ (Subword Splitting)

Dưới đây là hàm hậu xử lý để gộp các thực thể bị tách mảnh dựa trên vị trí `start` và `end` trong văn bản gốc.

In [ ]:
def merge_subword_entities(predictions):
    if not predictions:
        return []

    merged = []
    if len(predictions) > 0:
        curr = predictions[0].copy()

        for next_ent in predictions[1:]:
            # Nếu thực thể tiếp theo nối liền kề với thực thể hiện tại và cùng loại
            if next_ent['start'] == curr['end'] and next_ent['entity_group'] == curr['entity_group']:
                curr['word'] += next_ent['word']
                curr['end'] = next_ent['end']
                # Cập nhật score trung bình hoặc lấy max
                curr['score'] = (curr['score'] + next_ent['score']) / 2
            else:
                merged.append(curr)
                curr = next_ent.copy()
        merged.append(curr)
    return merged

# Chạy thử nghiệm
sample_text = "Cần tuyển chuyên gia về Synapse và kiến trúc Microservices."
raw_predictions = explicit_ner_pipeline(sample_text)

print("--- Trước khi gộp ---")
print(raw_predictions)

final_predictions = merge_subword_entities(raw_predictions)

print("\n--- Sau khi gộp thủ công ---")
for ent in final_predictions:
    print(f"Entity: {ent['word']}, Label: {ent['entity_group']}, Score: {ent['score']:.4f}")